# Phase 3: Dataset Cleaning and Candidate-Level Processing

This notebook processes the raw CSV data into a clean, ML-ready candidate dataset.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [2]:
DATA_DIR = r"backend\dataset"

df_people = pd.read_csv(os.path.join(DATA_DIR, "01_people.csv"))
df_abilities = pd.read_csv(os.path.join(DATA_DIR, "02_abilities.csv"))
df_education = pd.read_csv(os.path.join(DATA_DIR, "03_education.csv"))
df_experience = pd.read_csv(os.path.join(DATA_DIR, "04_experience.csv"))
df_person_skills = pd.read_csv(os.path.join(DATA_DIR, "05_person_skills.csv"))
df_skills = pd.read_csv(os.path.join(DATA_DIR, "06_skills.csv"))

## 3. Data Exploration

In [3]:
from IPython import display
print("People Data Info:")
display(df_people.info())
display(df_people.head(3))
print(f"\nUnique candidates: {df_people['person_id'].nunique()}")

People Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 54933 entries, 0 to 54932
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   person_id  54933 non-null  int64
 1   name       54819 non-null  str  
 2   email      1593 non-null   str  
 3   phone      1833 non-null   str  
 4   linkedin   8538 non-null   str  
dtypes: int64(1), str(4)
memory usage: 2.1 MB


None

,person_id,name,email,phone,linkedin
0,1,Database Administrator,NaN,NaN,NaN
1,2,Database Administrator,NaN,NaN,NaN
2,3,Oracle Database Administrator,NaN,NaN,NaN



Unique candidates: 54933


## 4. Cleaning

In [4]:
# Handle missing values
df_people.fillna({'email': '', 'phone': '', 'linkedin': ''}, inplace=True)
df_abilities.fillna({'ability': ''}, inplace=True)
df_education.fillna({'institution': '', 'program': '', 'start_date': '', 'location': ''}, inplace=True)
df_experience.fillna({'title': '', 'firm': '', 'start_date': '', 'end_date': '', 'location': ''}, inplace=True)
df_person_skills.fillna({'skill': ''}, inplace=True)
df_skills.fillna({'skill': ''}, inplace=True)

# Remove duplicates
df_people.drop_duplicates(inplace=True)
df_abilities.drop_duplicates(inplace=True)
df_education.drop_duplicates(inplace=True)
df_experience.drop_duplicates(inplace=True)
df_person_skills.drop_duplicates(inplace=True)
df_skills.drop_duplicates(inplace=True)

# Standardize text fields
df_abilities['ability'] = df_abilities['ability'].str.strip()
df_person_skills['skill'] = df_person_skills['skill'].str.strip().str.title()
df_experience['title'] = df_experience['title'].str.strip().str.title()

## 5. Merging & Feature Aggregation

In [5]:
# 5.1 Aggregate Skills
agg_skills = df_person_skills.groupby('person_id')['skill'].apply(lambda x: ', '.join(x.dropna().unique())).reset_index()
agg_skills.rename(columns={'skill': 'skills'}, inplace=True)

# 5.2 Aggregate Abilities
agg_abilities = df_abilities.groupby('person_id')['ability'].apply(lambda x: ', '.join(x.dropna().unique())).reset_index()
agg_abilities.rename(columns={'ability': 'abilities'}, inplace=True)

# 5.3 Aggregate Education
def agg_edu(group):
    edus = []
    for _, row in group.iterrows():
        edu = f"{row['program']} at {row['institution']}"
        if row['start_date']:
            edu += f" ({row['start_date']})"
        edus.append(edu.strip())
    return ' | '.join(edus)

agg_education = df_education.groupby('person_id').apply(agg_edu).reset_index(name='education')

# 5.4 Aggregate Experience & Extract Job Titles
def agg_exp(group):
    exps = []
    titles = []
    for _, row in group.iterrows():
        titles.append(str(row['title']))
        exp = f"{row['title']} at {row['firm']}"
        if row['start_date'] or row['end_date']:
            exp += f" ({row['start_date']} - {row['end_date']})"
        exps.append(exp.strip())
    
    # Filter out empty titles and remove duplicates
    titles = list(set([t for t in titles if t.strip()]))
    return pd.Series({'experience': ' | '.join(exps), 'job_titles': ', '.join(titles)})

agg_experience = df_experience.groupby('person_id').apply(agg_exp).reset_index()

## 6. Experience Calculation

In [6]:
def calculate_total_experience(group):
    total_years = 0
    current_year = 2024
    
    for _, row in group.iterrows():
        try:
            start_str = str(row['start_date'])
            end_str = str(row['end_date'])
            
            start_year_match = re.search(r'(19|20)\d{2}', start_str)
            
            if start_year_match:
                start_year = int(start_year_match.group(0))
                
                if 'present' in end_str.lower() or 'current' in end_str.lower():
                    end_year = current_year
                else:
                    end_year_match = re.search(r'(19|20)\d{2}', end_str)
                    if end_year_match:
                        end_year = int(end_year_match.group(0))
                    else:
                        end_year = start_year + 1 # Fallback assumption
                        
                years = end_year - start_year
                if 0 <= years < 50:
                    total_years += years
        except:
            pass
    return total_years

agg_exp_years = df_experience.groupby('person_id').apply(calculate_total_experience).reset_index(name='total_experience_years')

## 7. Final Candidate Dataset Creation

In [7]:
final_df = df_people.copy()

final_df = final_df.merge(agg_skills, on='person_id', how='left')
final_df = final_df.merge(agg_abilities, on='person_id', how='left')
final_df = final_df.merge(agg_education, on='person_id', how='left')
final_df = final_df.merge(agg_experience, on='person_id', how='left')
final_df = final_df.merge(agg_exp_years, on='person_id', how='left')

final_df.fillna({
    'skills': '',
    'abilities': '',
    'education': '',
    'experience': '',
    'job_titles': '',
    'total_experience_years': 0
}, inplace=True)

,person_id,name,email,phone,linkedin,skills,abilities,education,experience,job_titles,total_experience_years
0,1,Database Administrator,,,,"Database Administration, Database, Ms Sql Serv...","Installation and Building Server, Running Back...",Bachelor of Science at Lead City University (0...,Database Administrator at Family Private Care ...,Database Administrator,10
1,2,Database Administrator,,,,"Sql Server Management Studio, Visual Studio, S...","database management systems administration, de...",bsc in computer science at lagos state university,Database Administrator at Intercontinental Reg...,Database Administrator,3
2,3,Oracle Database Administrator,,,,"Databases, Oracle (4 Years), Oracle 10G, Sql, ...","Over 4+ years of Experience as Architecture, E...",Master of Computer Applications in Science and...,Oracle Database Administrator at Cognizant (06...,Oracle Database Administrator,10
3,4,Amazon Redshift Administrator and ETL Develope...,,,,Maintain Multiple Database Environments (Redsh...,"SQL management, PostgresSQL, Oracle, MySQL, mi...",Bachelor in Computer Science at University of ...,Amazon Redshift Administrator And Etl Develope...,Amazon Redshift Administrator And Etl Develope...,0
4,5,Scrum Master Scrum Master Scrum Master,,,,"Scrum, Agile Software Development, Product Bac...","Scrum Master, Agile software development, Prod...",at Virginia Commomwealth University (08/2013),Scrum Master at Quest Technologies (10/2015 - ...,"Junior Oracle Database Administrator, Scrum Ma...",7
...,...,...,...,...,...,...,...,...,...,...,...
54928,54929,Lead Python Developer,,,,"Django, Angular Js, Javascript, Jquery, Node.J...","Qualified Python Developer, Web Application De...",,Lead Python Developer at People's United Bank ...,"Python Developer, Sr. Python Developer, Web De...",17
54929,54930,Full Stack Python Developer,,,,"Python, Django, Aws, Angularjs, Bootstrap, Jav...","Over 7 years of IT Experience, Designing, deve...",,Full Stack Python Developer at Fresenius Medic...,"Java/ Python Developer, Sr. Python Developer, ...",14
54930,54931,Eli Lilly,,,,"Python 2.7, Html5, Css3, Ajax, Json, Jquery, A...","Developing and designing Web Based, Multi-tier...",,Sr. Python Developer at Eli Lilly (08/2017 - P...,"Python Developer, Sr. Python Developer, Java D...",14
54931,54932,Python Developer,,,,"Python 3.1X, Pyquery, Pyqt, Django, Angular.Js...","Python Developer, Web/Application Developer, A...",,Python Developer at Intuit (08/2015 - Present)...,"Python Developer, Software Developer",15


## 8. Feature Engineering (NLP Text Creation)

In [8]:
def create_combined_profile(row):
    parts = []
    if row['skills']: parts.append(f"Skills: {row['skills']}")
    if row['abilities']: parts.append(f"Abilities: {row['abilities']}")
    if row['education']: parts.append(f"Education: {row['education']}")
    if row['experience']: parts.append(f"Experience: {row['experience']}")
    if row['job_titles']: parts.append(f"Job Titles: {row['job_titles']}")
    
    return "\n".join(parts)

final_df['combined_profile_text'] = final_df.apply(create_combined_profile, axis=1)

display(final_df.head(2))

,person_id,name,email,phone,linkedin,skills,abilities,education,experience,job_titles,total_experience_years,combined_profile_text
0,1,Database Administrator,,,,"Database Administration, Database, Ms Sql Serv...","Installation and Building Server, Running Back...",Bachelor of Science at Lead City University (0...,Database Administrator at Family Private Care ...,Database Administrator,10,"Skills: Database Administration, Database, Ms ..."
1,2,Database Administrator,,,,"Sql Server Management Studio, Visual Studio, S...","database management systems administration, de...",bsc in computer science at lagos state university,Database Administrator at Intercontinental Reg...,Database Administrator,3,"Skills: Sql Server Management Studio, Visual S..."


## 9. Export

In [9]:
OUTPUT_FILE = os.path.join(DATA_DIR, "processed_candidates.csv")
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Successfully saved processed dataset to: {OUTPUT_FILE}")

Successfully saved processed dataset to: backend\dataset\processed_candidates.csv


## 10. Validation

In [10]:
print("--- Validation Report ---")
print(f"Total unique candidates before processing: {df_people['person_id'].nunique()}")
print(f"Total candidates after processing: {final_df['person_id'].nunique()}")
print(f"\nMissing values in final dataset:\n{final_df.isnull().sum()}")
print(f"\nDuplicates in final dataset: {final_df.duplicated().sum()}")
print("\nSample processed rows (first 5):")
display(final_df[['person_id', 'name', 'skills', 'job_titles', 'total_experience_years']].head(5))

--- Validation Report ---
Total unique candidates before processing: 54933
Total candidates after processing: 54933

Missing values in final dataset:
person_id                   0
name                      114
email                       0
phone                       0
linkedin                    0
skills                      0
abilities                   0
education                   0
experience                  0
job_titles                  0
total_experience_years      0
combined_profile_text       0
dtype: int64

Duplicates in final dataset: 0

Sample processed rows (first 5):


,person_id,name,skills,job_titles,total_experience_years
0,1,Database Administrator,"Database Administration, Database, Ms Sql Serv...",Database Administrator,10
1,2,Database Administrator,"Sql Server Management Studio, Visual Studio, S...",Database Administrator,3
2,3,Oracle Database Administrator,"Databases, Oracle (4 Years), Oracle 10G, Sql, ...",Oracle Database Administrator,10
3,4,Amazon Redshift Administrator and ETL Develope...,Maintain Multiple Database Environments (Redsh...,Amazon Redshift Administrator And Etl Develope...,0
4,5,Scrum Master Scrum Master Scrum Master,"Scrum, Agile Software Development, Product Bac...","Junior Oracle Database Administrator, Scrum Ma...",7
